<a href="https://www.kaggle.com/code/avikdas567/grouped-time-series-forecasting-with-lightgbm?scriptVersionId=306824286" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm

import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit

# PATHS

TRAIN_PATH = "/kaggle/input/competitions/ts-forecasting/train.parquet"
TEST_PATH  = "/kaggle/input/competitions/ts-forecasting/test.parquet"

# METRIC

def weighted_rmse_score(y_true, y_pred, w):
    denom = np.sum(w * y_true ** 2)
    ratio = np.sum(w * (y_true - y_pred) ** 2) / denom
    ratio = np.clip(ratio, 0, 1)
    return np.sqrt(1.0 - ratio)

# LOAD DATA

print("Loading data...")
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

# SETUP

ID_COL = "id"
TIME_COL = "ts_index"
GROUP_COLS = ["code", "sub_code", "sub_category", "horizon"]

FEATURES = [c for c in train.columns if c.startswith("feature_")]

# Detect target
KNOWN_COLS = set([ID_COL, TIME_COL, "code", "sub_code", "sub_category", "horizon", "weight"] + FEATURES)
TARGET = [c for c in train.columns if c not in KNOWN_COLS][0]

print("TARGET:", TARGET)

# SORT

train = train.sort_values(GROUP_COLS + [TIME_COL]).reset_index(drop=True)
test  = test.sort_values(GROUP_COLS + [TIME_COL]).reset_index(drop=True)

# FAST FEATURES

def create_lag_features(df):
    for lag in [1,2]:
        df[f"lag_{lag}"] = df.groupby(GROUP_COLS)[TARGET].shift(lag)
    return df

def create_rolling_features(df):
    df["roll_mean_3"] = df.groupby(GROUP_COLS)[TARGET].shift(1).rolling(3).mean()
    return df

print("Creating features...")
train = create_lag_features(train)
train = create_rolling_features(train)

train = train.dropna().reset_index(drop=True)

# FEATURES

EXTRA_FEATURES = ["lag_1", "lag_2", "roll_mean_3"]
ALL_FEATURES = FEATURES + EXTRA_FEATURES

# VALIDATION

tscv = TimeSeriesSplit(n_splits=2)

X = train[ALL_FEATURES]
y = train[TARGET]
w = train["weight"]

oof = np.zeros(len(train))

# MODEL

params = {
    "objective": "regression",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "verbosity": -1
}

print("Training...")

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X)):
    
    print(f"\nFold {fold+1}")
    
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    w_tr, w_val = w.iloc[tr_idx], w.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**params, n_estimators=400)
    
    model.fit(
        X_tr, y_tr,
        sample_weight=w_tr,
        eval_set=[(X_val, y_val)],
        eval_sample_weight=[w_val],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )
    
    preds = model.predict(X_val)
    oof[val_idx] = preds
    
    score = weighted_rmse_score(y_val.values, preds, w_val.values)
    print("Fold Score:", score)

# CV
cv_score = weighted_rmse_score(y.values, oof, w.values)
print("\nCV SCORE:", cv_score)

# FINAL MODEL

print("\nTraining final model...")

final_model = lgb.LGBMRegressor(**params, n_estimators=300)
final_model.fit(X, y, sample_weight=w)

# TEST FEATURES

print("\nProcessing test...")

combined = pd.concat([train, test], axis=0)
combined = combined.sort_values(GROUP_COLS + [TIME_COL]).reset_index(drop=True)

combined = create_lag_features(combined)
combined = create_rolling_features(combined)

test_processed = combined.iloc[len(train):].copy()
test_processed[ALL_FEATURES] = test_processed[ALL_FEATURES].fillna(0)

# PREDICTION

print("Predicting...")
test_preds = final_model.predict(test_processed[ALL_FEATURES])

# SUBMISSION

submission = pd.DataFrame({
    "id": test_processed[ID_COL],
    "prediction": test_preds
})

submission.to_csv("submission.csv", index=False)

print("'submission.csv' created.")
display(submission.head())

Loading data...
TARGET: y_target
Creating features...
Training...

Fold 1
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 8.16058e-06
Early stopping, best iteration is:
[63]	valid_0's l2: 4.25688e-06
Fold Score: 0.8101105430076302

Fold 2
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 4.20336e-06
Early stopping, best iteration is:
[63]	valid_0's l2: 3.18783e-06
Fold Score: 0.7354398685497706

CV SCORE: 0.6687746405257226

Training final model...

Processing test...
Predicting...
'submission.csv' created.


,id,prediction
3748554,QAQDDTPJ__KZSDAYZT__PHHHVYZI__1__2644,29.932778
3748555,QAQDDTPJ__KZSDAYZT__PHHHVYZI__1__2645,4.538899
3748556,QAQDDTPJ__KZSDAYZT__PHHHVYZI__1__2646,4.386468
3748557,QAQDDTPJ__KZSDAYZT__PHHHVYZI__1__2647,-2.615560
3748558,QAQDDTPJ__KZSDAYZT__PHHHVYZI__1__2648,-6.525098
